# Dynamic Entité Incident Table
Upload a JSON file of incidents. Use the dropdown to select an entity and view their incident repartition by category.

In [ ]:
import pandas as pd
import json
import re
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML

!pip install ipywidgets -q

def load_json_to_df(path):
    for encoding in ('utf-8', 'latin-1'):
        try:
            with open(path, encoding=encoding) as f:
                data = json.load(f)
            return pd.json_normalize(data)
        except Exception:
            continue
    raise ValueError(f'Unable to load JSON file: {path}')

def normalize_columns(df):
    mapping = {}
    for col in df.columns:
        normalized = re.sub(r'[^a-z0-9]', '', str(col).lower())
        if normalized in ('client', 'clientname', 'tenant', 'societe', 'company', 'entite', 'entité'):
            mapping[col] = 'Entité'
        elif normalized in ('serveur', 'server', 'host', 'hostname', 'apparail'):
            mapping[col] = 'Serveur'
        elif normalized in ('titre', 'title', 'subject', 'objet', 'description', 'summary'):
            mapping[col] = 'Titre'
    df = df.rename(columns=mapping)
    for target in ('Entité', 'Serveur', 'Titre'):
        if target not in df.columns:
            df[target] = pd.NA
    return df

def categorize_row(row):
    titre = str(row.get('Titre', '') or '')
    serveur = str(row.get('Serveur', '') or '')
    combined = f'{titre} {serveur}'
    if re.search(r'ntnx|nutanix', combined, flags=re.I):
        return 'Nutanix'
    if re.search(r'cpu', titre, flags=re.I):
        return 'CPU Issue'
    if re.search(r'memory|ram', titre, flags=re.I):
        return 'Memory Issue'
    if re.search(r'disk|storage', titre, flags=re.I):
        return 'Disk'
    if re.search(r'network|nic|link', titre, flags=re.I):
        return 'Network'
    return 'OS / Autres'

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   --------------------- ---------------- 524.3/914.9 kB 445.1 kB/s eta 0:00:01
   --------------------- ---------------- 524.3/914.9 kB 445.1 kB/s eta 0:00:01
   --------------------- ---------------- 524.3/914.9 kB 445.1 kB/s eta 0:00:01
   --------------------- ---------------- 524.3/914.9 kB 445.1 kB/s eta 0:00:01
   --------------------- ---------------- 524.3/914.9 kB 445.1 kB/s eta 0:00:01
   -------------------------------- ----- 786.4/914.9 kB 330.0 kB/s eta 0:00:01
   --------------

In [ ]:
# Enter the path to your JSON file and run this cell
import os

json_path = "Synthèse DC - Incident.json"  # Change this to your JSON file path

if os.path.exists(json_path):
    df = load_json_to_df(json_path)
    df = normalize_columns(df)
    df['Categorie'] = df.apply(categorize_row, axis=1)
    entities = sorted(df['Entité'].dropna().unique())
    print(f"✅ Loaded {len(df)} incidents")
    print(f"📋 Found {len(entities)} entités")
    entity_dropdown.options = entities
    entity_dropdown.value = entities[0] if entities else None
    entity_dropdown.df = df
else:
    print(f"❌ File not found: {json_path}")
    print("Please update the json_path variable above with your JSON file path")

Note: you may need to restart the kernel to use updated packages.


FileUpload(value=(), accept='.json', description='Upload')

In [ ]:
# Dropdown and dynamic table
entity_dropdown = widgets.Dropdown(description='Entité:', options=[])
output = widgets.Output()

def update_table(change):
    output.clear_output()
    df = getattr(entity_dropdown, 'df', None)
    if df is None or not entity_dropdown.value:
        return
    filtered = df[df['Entité'] == entity_dropdown.value]
    category_order = [
        'CPU Issue', 'Memory Issue', 'Disk', 'Network', 'Nutanix', 'OS / Autres'
    ]
    display_map = {
        'CPU Issue': 'CPU Issue',
        'Memory Issue': 'Memory Issue',
        'OS / Autres': 'OS service',
        'Nutanix': 'Nutanix Issue',
        'Disk': 'VM availability / Disk / MSSQL / autres',
        'Network': 'Network',
    }
    counts = filtered['Categorie'].value_counts().reindex(category_order, fill_value=0)
    total = counts.sum()
    table_df = (
        counts.reset_index(name='Volume')
        .assign(Categorie=lambda df: df['index'].map(display_map).fillna(df['index']))
        .assign(Part_du_total=lambda df: (df['Volume'] / total * 100).round(0).astype(int).astype(str) + ' %')
        .loc[lambda df: df['Categorie'].isin([
            'CPU Issue', 'Memory Issue', 'OS service', 'Nutanix Issue', 'VM availability / Disk / MSSQL / autres'
        ])]
        .loc[:, ['Categorie', 'Volume', 'Part_du_total']]
    )
    with output:
        display(HTML(table_df.to_html(index=False, escape=False)))

entity_dropdown.observe(update_table, names='value')
display(entity_dropdown, output)